> **Versión corregida (v2.12, ejecución verificada)** — generada a partir de `00_setup_colab_reinicio.ipynb (Drive/Colab Notebooks)` sin sobrescribirlo.
>
> Cambios aplicados:
> - Celda de inicialización sustituida por el bootstrap portable: las rutas fijas `/content/drive/MyDrive/PIDA-RL-Diabetes` ahora dependen de `PROJECT_ROOT`.
> - Renombrado semántico: el antiguo `PROJECT_ROOT` (copia del repositorio) pasa a `REPO_ROOT`; el antiguo `PERSISTENT_ROOT` (Drive) pasa a `PROJECT_ROOT` (se conserva `PERSISTENT_ROOT` como alias).
> - El clonado usa la rama `feature/entorno-capacidad-v2`; antes clonaba `main`, que no contiene `src/` V2.
> - Montaje de Drive movido al bootstrap; fuera de Colab no se exige Drive.
> - Instalación de dependencias: solo paquetes faltantes.
> - Checklist final: `Drive montado` solo se exige en Colab; se añade `PROJECT_ROOT disponible`.
>
> Rutas: `PROJECT_ROOT` = datos/modelos/resultados (Drive en Colab, o `PIDA_PROJECT_ROOT`); `REPO_ROOT` = código. Modo prueba: `PIDA_MODO_PRUEBA=1` (cohorte pequeña y pocos episodios).


# PIDA-RL Diabetes — 00. Setup reproducible en Google Colab

Este es el **único notebook de preparación del entorno**. Su propósito es dejar una sesión de Google Colab lista para ejecutar el proyecto de forma reproducible, sin entrenar modelos, modificar datos, ni realizar commits automáticos.

## Principios de esta nueva etapa

- Google Drive es el almacenamiento persistente de datos, modelos, resultados y figuras.
- `/content/pida-rl-datos-publicos` es una copia temporal del repositorio durante la sesión de Colab.
- Las rutas se resuelven de forma robusta para evitar depender de si Drive quedó montado en `/content/drive` o `/content/gdrive`.
- Este notebook **no falla** si aún no existe una cohorte o un resultado: esos artefactos se crearán en los notebooks posteriores.
- Nunca escribas, imprimas o guardes tokens, contraseñas o credenciales en celdas, salidas o commits.


## 1. Parámetros del proyecto

Modifica únicamente `REPO_URL` si cambia la ubicación del repositorio. Las rutas derivadas se usarán en todos los notebooks posteriores.

In [ ]:
# === Bootstrap portable: PROJECT_ROOT (corrección v2.12) ===
# PROJECT_ROOT = raíz persistente de datos, modelos y resultados.
# REPO_ROOT    = copia local del código (repositorio GitHub).
# Prioridad: variables de entorno PIDA_PROJECT_ROOT / PIDA_REPO_ROOT,
# después Google Colab + Drive y, por último, ejecución local.
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import importlib
import importlib.util
import subprocess

PROJECT_NAME = 'pida-rl-datos-publicos'

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = os.environ.get('PIDA_BRANCH', 'feature/entorno-capacidad-v2')
PROJECT_FOLDER_NAME = 'PIDA-RL-Diabetes'
IN_COLAB = 'google.colab' in sys.modules

DRIVE_ROOT = None
if os.environ.get('PIDA_PROJECT_ROOT'):
    PROJECT_ROOT = Path(os.environ['PIDA_PROJECT_ROOT']).expanduser().resolve()
elif IN_COLAB:
    DRIVE_ROOT = next(
        (p for p in [Path('/content/drive'), Path('/content/gdrive')] if (p / 'MyDrive').is_dir()),
        None,
    )
    if DRIVE_ROOT is None:
        from google.colab import drive
        mount_point = Path('/content/drive')
        if mount_point.exists() and any(mount_point.iterdir()):
            mount_point = Path('/content/gdrive')
        drive.mount(str(mount_point), force_remount=False)
        DRIVE_ROOT = mount_point
    PROJECT_ROOT = DRIVE_ROOT / 'MyDrive' / PROJECT_FOLDER_NAME
else:
    PROJECT_ROOT = (Path.cwd() / PROJECT_FOLDER_NAME).resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


def _buscar_repo(inicio):
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'src' / 'environment' / 'diabetes_followup_env.py').is_file():
            return candidato
    return None


if os.environ.get('PIDA_REPO_ROOT'):
    REPO_ROOT = Path(os.environ['PIDA_REPO_ROOT']).expanduser().resolve()
else:
    REPO_ROOT = _buscar_repo(Path.cwd().resolve()) or (
        Path('/content/pida-rl-datos-publicos') if IN_COLAB
        else Path.cwd().resolve() / 'pida-rl-datos-publicos'
    )
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])
elif not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError(f'REPO_ROOT no contiene la carpeta src: {REPO_ROOT}')
elif IN_COLAB and (REPO_ROOT / '.git').is_dir():
    # En Colab se alinea la copia temporal con la rama de trabajo (sin push ni commit).
    for args in (['fetch', 'origin', BRANCH], ['checkout', BRANCH], ['pull', '--ff-only', 'origin', BRANCH]):
        subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=False)

SRC_ROOT = REPO_ROOT / 'src'
for ruta in (str(REPO_ROOT), str(SRC_ROOT)):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)
os.chdir(REPO_ROOT)

# Estructura persistente común (antes: rutas fijas /content/drive/MyDrive/PIDA-RL-Diabetes).
PERSISTENT_ROOT = PROJECT_ROOT  # alias de compatibilidad con los notebooks originales
MYDRIVE_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
INTERIM_DATA_DIR = DATA_ROOT / 'interim'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'
SPECIFICATIONS_DIR = DATA_ROOT / 'specifications'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'

# Modo prueba: cohorte pequeña y pocos episodios (PIDA_MODO_PRUEBA=1).
MODO_PRUEBA = os.environ.get('PIDA_MODO_PRUEBA', '0') == '1'
N_COHORTE_PRUEBA = int(os.environ.get('PIDA_N_COHORTE', '120'))
SEMILLA_MUESTRA_PRUEBA = 2026
RESULTS_V2_DIR = RESULTS_DIR / ('capacidad_v2_prueba' if MODO_PRUEBA else 'capacidad_v2')


def muestra_cohorte_prueba(cohorte, n=None, semilla=SEMILLA_MUESTRA_PRUEBA):
    """Devuelve (cohorte, factor). En modo prueba toma una submuestra
    estratificada por categoria_riesgo (bajo/medio/alto); no crea columnas."""
    n = N_COHORTE_PRUEBA if n is None else int(n)
    if not MODO_PRUEBA or len(cohorte) <= n:
        return cohorte, 1.0
    fraccion = n / len(cohorte)
    muestra = (
        cohorte.groupby('categoria_riesgo', group_keys=False)
        .sample(frac=fraccion, random_state=semilla)
        .sort_values('FOLIO_INT')
        .reset_index(drop=True)
    )
    return muestra, len(muestra) / len(cohorte)


def escalar_capacidad(capacidad, factor):
    """Mantiene la presión de capacidad (recursos/persona) al reducir la cohorte."""
    return capacidad if factor >= 1.0 else max(1, int(round(capacidad * factor)))

print('Entorno:', 'Google Colab' if IN_COLAB else 'local')
print('PROJECT_ROOT (datos/resultados):', PROJECT_ROOT)
print('REPO_ROOT (código):', REPO_ROOT)
print('Modo prueba:', MODO_PRUEBA)

# Déjalo como None para respetar la rama actual del repositorio.
WORK_BRANCH = None


## 2. Montar Google Drive sin conflictos

Esta celda reutiliza Drive si ya está montado. Si `/content/drive` contiene archivos locales y no está montado, usa `/content/gdrive` como punto alternativo.

In [ ]:
# Drive se monta en el bootstrap solo cuando se ejecuta en Colab.
print('Google Drive:', DRIVE_ROOT or 'no aplica (ejecución local)')
print('Raíz persistente del proyecto (PROJECT_ROOT):', PROJECT_ROOT)


## 3. Crear estructura persistente

Se crean solamente carpetas. No se crean datos sintéticos, cohortes vacías ni modelos ficticios.

In [ ]:
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
INTERIM_DATA_DIR = DATA_ROOT / 'interim'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
LOGS_DIR = PROJECT_ROOT / 'logs'
EXPORTS_DIR = PROJECT_ROOT / 'exports'

persistent_dirs = [
    RAW_DATA_DIR, INTERIM_DATA_DIR, PROCESSED_DATA_DIR,
    MODELS_DIR, TABLES_DIR, FIGURES_DIR, LOGS_DIR, EXPORTS_DIR
]

for directory in persistent_dirs:
    directory.mkdir(parents=True, exist_ok=True)

print('Carpetas persistentes listas:')
for directory in persistent_dirs:
    print(' -', directory)


## 4. Instalar dependencias

Las dependencias se instalan en la sesión actual. Si Colab indica `Requirement already satisfied`, no es un error.

In [ ]:
# Instala solo lo que falte (pip -> módulo importable).
PACKAGES = {
    'numpy': 'numpy', 'pandas': 'pandas', 'scikit-learn': 'sklearn',
    'matplotlib': 'matplotlib', 'seaborn': 'seaborn', 'pyreadstat': 'pyreadstat',
    'gymnasium': 'gymnasium', 'stable-baselines3': 'stable_baselines3',
    'streamlit': 'streamlit', 'plotly': 'plotly', 'pyyaml': 'yaml', 'joblib': 'joblib',
}
faltantes = [pip for pip, mod in PACKAGES.items() if importlib.util.find_spec(mod) is None]
if faltantes:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *faltantes])
print('Dependencias disponibles. Instaladas en esta sesión:', faltantes or 'ninguna')


## 5. Clonar o reutilizar el repositorio

El repositorio vive temporalmente en `/content`. Esta celda no hace `pull`, `push`, `commit` ni cambia de rama automáticamente.

In [ ]:
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])
    print('Repositorio clonado.')
elif not (REPO_ROOT / '.git').is_dir():
    raise RuntimeError(
        f'La ruta existe pero no es un repositorio Git: {REPO_ROOT}. '
        'Renombra o elimina esa carpeta local antes de continuar.'
    )
else:
    print('Repositorio local encontrado; no se vuelve a clonar.')

os.chdir(REPO_ROOT)
print('Directorio actual:', Path.cwd())


## 6. Estado de Git y código fuente

La identidad de Git es opcional para ejecutar el proyecto. Nunca configures tokens mediante `git config`.

In [ ]:
def run_git(*args, check=True):
    result = subprocess.run(
        ['git', *args],
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
        check=check
    )
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    return result

run_git('status', '--short')
current_branch = run_git('branch', '--show-current').stdout.strip()
print('Rama activa:', current_branch or '(sin rama detectada)')
run_git('log', '--oneline', '-3', check=False)

if WORK_BRANCH is not None and current_branch != WORK_BRANCH:
    raise RuntimeError(
        f'La rama activa es {current_branch!r}; se esperaba {WORK_BRANCH!r}. '
        'Cámbiala manualmente antes de ejecutar el resto del flujo.'
    )

SRC_ROOT = REPO_ROOT / 'src'
SRC_ROOT.mkdir(parents=True, exist_ok=True)

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('SRC_ROOT agregado a sys.path:', SRC_ROOT)


## 7. Validar rutas de entrada y salida

El setup solo reporta el estado de los artefactos. La cohorte se crea en el notebook 03; los modelos se entrenan después de validar el MDP.

In [ ]:
RAW_PATTERNS = ('*.sav', '*.csv', '*.xlsx', '*.xls', '*.parquet')
raw_files = []
for pattern in RAW_PATTERNS:
    raw_files.extend(RAW_DATA_DIR.rglob(pattern))

COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'

print('Archivos fuente detectados:', len(raw_files))
for path in sorted(raw_files)[:20]:
    print(' -', path.name)
if len(raw_files) > 20:
    print(' - ...')

print('Cohorte procesada:', COHORT_PATH)
print('Existe cohorte:', COHORT_PATH.is_file())
print('Directorio de modelos:', MODELS_DIR)
print('Directorio de tablas:', TABLES_DIR)
print('Directorio de figuras:', FIGURES_DIR)


## 8. Diagnóstico de versiones

Registra estas versiones en la bitácora del proyecto cuando ejecutes experimentos reproducibles.

In [ ]:
import numpy as np
import pandas as pd
import gymnasium
import stable_baselines3
import pyreadstat
import sklearn

versions = {
    'python': sys.version.split()[0],
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit-learn': sklearn.__version__,
    'gymnasium': gymnasium.__version__,
    'stable-baselines3': stable_baselines3.__version__,
    'pyreadstat': pyreadstat.__version__
}

for package, version in versions.items():
    print(f'{package}: {version}')


## 9. Flujo de ejecución oficial

El orden evita evaluar modelos antes de entrenarlos y evita ejecutar análisis de equidad antes de contar con una evaluación conjunta.

1. `01_fuentes_y_carga.ipynb` — carga y trazabilidad de fuentes públicas.
2. `02_entendimiento_datos.ipynb` — calidad, estructura y exploración de datos.
3. `03_preparacion_cohorte.ipynb` — cohorte final y validación de valores faltantes/no finitos.
4. `04_definicion_mdp.ipynb` — entorno Gymnasium, estado, acciones, recompensas y `check_env`.
5. `05_entrenamiento_dqn.ipynb` — entrenamiento y persistencia de DQN.
6. `06_entrenamiento_q_learning.ipynb` — entrenamiento y persistencia de Q-learning tabular.
7. `07_entrenamiento_ppo.ipynb` — validación y entrenamiento de PPO.
8. `08_evaluacion_modelos.ipynb` — evaluación homogénea: aleatoria, reglas, DQN, Q-learning y PPO.
9. `09_equidad_y_subgrupos.ipynb` — cobertura y brechas por riesgo, sexo y edad.
10. `10_resultados_finales.ipynb` — tablas, visualizaciones y conclusiones técnicas.
11. `11_dashboard_streamlit.ipynb` — generación, prueba y despliegue local de la app Streamlit.
12. `12_documentacion_reproducibilidad.ipynb` — README, CRISP-DM, registro de experimentos y checklist de entrega.

### Convención de artefactos persistentes

- Cohorte: `data/processed/cohorte_diabetes.csv`
- Modelos: `models/`
- Tablas de evaluación: `results/tables/`
- Figuras: `results/figures/`
- Bitácoras y TensorBoard: `logs/`

Los notebooks posteriores deben obtener las rutas desde esta misma convención y verificar la existencia de cada insumo antes de usarlo.

## 10. Checklist final del setup

Ejecuta esta celda antes de continuar. Si la cohorte aún no existe, es normal: se creará en el notebook 03.

In [ ]:
checks = {
    'Drive montado (solo Colab)': (DRIVE_ROOT is not None) if IN_COLAB else True,
    'PROJECT_ROOT disponible': PROJECT_ROOT.is_dir(),
    'Raíz persistente': PROJECT_ROOT.is_dir(),
    'Repositorio Git': (REPO_ROOT / '.git').is_dir(),
    'Código fuente src': SRC_ROOT.is_dir(),
    'Datos raw': RAW_DATA_DIR.is_dir(),
    'Datos processed': PROCESSED_DATA_DIR.is_dir(),
    'Modelos': MODELS_DIR.is_dir(),
    'Tablas': TABLES_DIR.is_dir(),
    'Figuras': FIGURES_DIR.is_dir(),
}

for name, ok in checks.items():
    print(('✓' if ok else '✗'), name)

assert checks['Drive montado (solo Colab)']
assert checks['PROJECT_ROOT disponible']
assert checks['Repositorio Git']
assert checks['Código fuente src']

print('\nSetup terminado. Continúa con 01_fuentes_y_carga.ipynb.')
